In [1]:
from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

try:
    from sklearn.neighbors import BallTree
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [2]:
REPO_ROOT = Path.cwd()

RAW_PERF_PATH = REPO_ROOT / "data_raw" / "store_performance_2018to2022.csv"
STORE_INFO_PATH = REPO_ROOT / "data_raw" / "store_info.csv"

WEATHER_CANDIDATES = [
    REPO_ROOT / "data_processed" / "weather_features_2018_2022.csv",
    REPO_ROOT / "data_processed" / "weather_allstores.parquet",
    REPO_ROOT / "data_processed" / "weather_allstores.csv",
]

OUT_FEATURES_PATH = REPO_ROOT / "data_processed" / "sprint3_features_weather_impact_ready.csv"
OUT_WEATHER_IMPACT_PATH = REPO_ROOT / "data_processed" / "weather_impact_ci_2022_sprint3.csv"
OUT_NEIGHBOR_MAP_PATH = REPO_ROOT / "data_processed" / "nearby_store_map_top5.csv"

In [3]:
TARGET_COL = "oc_count"  
VALID_START = pd.Timestamp("2022-01-01")
NEIGHBOR_K = 5            
MAX_NEIGHBOR_DISTANCE_KM = 80.0  
BASELINE_LABEL = "Clear / Mild (baseline)"


In [4]:
def load_first_existing_weather(path_candidates: Sequence[Path]) -> pd.DataFrame:
    for path in path_candidates:
        if path.exists():
            print(f"[INFO] Loading weather from: {path}")
            if path.suffix.lower() == ".parquet":
                return pd.read_parquet(path)
            if path.suffix.lower() == ".csv":
                return pd.read_csv(path)
    raise FileNotFoundError(
        "No weather file found. Checked: " + ", ".join(str(p) for p in path_candidates)
    )

In [5]:
def standardize_date_col(df: pd.DataFrame, preferred_names: Sequence[str]) -> pd.DataFrame:
    df = df.copy()
    cols_lower = {c.lower(): c for c in df.columns}
    for name in preferred_names:
        if name.lower() in cols_lower:
            actual = cols_lower[name.lower()]
            df = df.rename(columns={actual: "date"})
            df["date"] = pd.to_datetime(df["date"])
            return df
    raise KeyError(f"Could not find a date column from candidates: {preferred_names}")

In [6]:
def standardize_store_id_col(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    candidates = ["store_id", "store", "storeid"]
    cols_lower = {c.lower(): c for c in df.columns}
    for name in candidates:
        if name in cols_lower:
            actual = cols_lower[name]
            df = df.rename(columns={actual: "store_id"})
            return df
    raise KeyError("Could not find store_id column.")

In [7]:
def find_column(df: pd.DataFrame, candidates: Sequence[str], required: bool = True) -> Optional[str]:
    cols_lower = {c.lower(): c for c in df.columns}
    for name in candidates:
        if name.lower() in cols_lower:
            return cols_lower[name.lower()]
    if required:
        raise KeyError(f"Missing required column. Tried: {candidates}")
    return None

In [8]:
def ensure_numeric(df: pd.DataFrame, cols: Iterable[str]) -> pd.DataFrame:
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def pct_change_vs_baseline(group_mean: float, baseline_mean: float) -> float:
    if pd.isna(group_mean) or pd.isna(baseline_mean) or baseline_mean == 0:
        return np.nan
    return (group_mean - baseline_mean) / baseline_mean * 100.0


def mean_ci_from_series(x: pd.Series, z: float = 1.96) -> Tuple[float, float, float]:
    x = pd.to_numeric(x, errors="coerce").dropna()
    n = len(x)
    if n == 0:
        return np.nan, np.nan, np.nan
    mean = x.mean()
    if n == 1:
        return mean, np.nan, np.nan
    se = x.std(ddof=1) / math.sqrt(n)
    return mean, mean - z * se, mean + z * se

In [9]:
from pathlib import Path

# If notebook is inside repo/notebooks, go one level up to repo root
REPO_ROOT = Path.cwd().parent

RAW_PERF_PATH = REPO_ROOT / "data_raw" / "store_performance_2018to2022.csv"
STORE_INFO_PATH = REPO_ROOT / "data_raw" / "store_info.csv"

WEATHER_CANDIDATES = [
    REPO_ROOT / "data_processed" / "weather_features_2018_2022.csv",
    REPO_ROOT / "data_processed" / "weather_allstores.parquet",
    REPO_ROOT / "data_processed" / "weather_allstores.csv",
]

OUT_FEATURES_PATH = REPO_ROOT / "data_processed" / "sprint3_features_weather_impact_ready.csv"
OUT_WEATHER_IMPACT_PATH = REPO_ROOT / "data_processed" / "weather_impact_ci_2022_sprint3.csv"
OUT_NEIGHBOR_MAP_PATH = REPO_ROOT / "data_processed" / "nearby_store_map_top5.csv"

In [10]:
print("CWD:", Path.cwd())
print("REPO_ROOT:", REPO_ROOT)
print("RAW_PERF_PATH exists?", RAW_PERF_PATH.exists(), RAW_PERF_PATH)
print("STORE_INFO_PATH exists?", STORE_INFO_PATH.exists(), STORE_INFO_PATH)

for p in WEATHER_CANDIDATES:
    print("Weather candidate:", p.exists(), p)

CWD: /Users/harshini/GenAI-Weather-Based-Store-Analytics/notebooks
REPO_ROOT: /Users/harshini/GenAI-Weather-Based-Store-Analytics
RAW_PERF_PATH exists? True /Users/harshini/GenAI-Weather-Based-Store-Analytics/data_raw/store_performance_2018to2022.csv
STORE_INFO_PATH exists? True /Users/harshini/GenAI-Weather-Based-Store-Analytics/data_raw/store_info.csv
Weather candidate: True /Users/harshini/GenAI-Weather-Based-Store-Analytics/data_processed/weather_features_2018_2022.csv
Weather candidate: True /Users/harshini/GenAI-Weather-Based-Store-Analytics/data_processed/weather_allstores.parquet
Weather candidate: False /Users/harshini/GenAI-Weather-Based-Store-Analytics/data_processed/weather_allstores.csv


In [12]:
perf = pd.read_csv(RAW_PERF_PATH)
store_info = pd.read_csv(STORE_INFO_PATH)
weather = load_first_existing_weather(WEATHER_CANDIDATES)

[INFO] Loading weather from: /Users/harshini/GenAI-Weather-Based-Store-Analytics/data_processed/weather_features_2018_2022.csv


In [15]:
weather = standardize_store_id_col(weather)
weather = standardize_date_col(weather, ["date", "time", "day", "invoice_date"])

print(weather.columns.tolist()[:20])

['store_id', 'date', 'tavg', 'tmin', 'tmax', 'prcp', 'snow', 'wspd', 'tavg_lag1', 'tavg_lag3', 'tavg_lag7', 'tmin_lag1', 'tmin_lag3', 'tmin_lag7', 'tmax_lag1', 'tmax_lag3', 'tmax_lag7', 'prcp_lag1', 'prcp_lag3', 'prcp_lag7']


In [16]:
print(type(weather))
print(weather.shape)
print(weather.columns.tolist()[:20])

<class 'pandas.core.frame.DataFrame'>
(801175, 34)
['store_id', 'date', 'tavg', 'tmin', 'tmax', 'prcp', 'snow', 'wspd', 'tavg_lag1', 'tavg_lag3', 'tavg_lag7', 'tmin_lag1', 'tmin_lag3', 'tmin_lag7', 'tmax_lag1', 'tmax_lag3', 'tmax_lag7', 'prcp_lag1', 'prcp_lag3', 'prcp_lag7']


In [17]:
weather = weather.copy()

# Likely weather columns from Meteostat-style daily data or engineered weather files.
COL_TAVG = find_column(weather, ["tavg", "avg_temp_c", "temp_avg", "temperature_avg"], required=False)
COL_TMIN = find_column(weather, ["tmin", "min_temp_c", "temperature_min"], required=False)
COL_TMAX = find_column(weather, ["tmax", "max_temp_c", "temperature_max"], required=False)
COL_PRCP = find_column(weather, ["prcp", "precip", "precipitation", "precip_mm"], required=False)
COL_SNOW = find_column(weather, ["snow", "snow_cm", "snowfall", "snow_depth"], required=False)
COL_WSPD = find_column(weather, ["wspd", "wind_speed", "wind_kph", "wind_speed_kph"], required=False)
COL_WDIR = find_column(weather, ["wdir", "wind_dir", "wind_direction"], required=False)
COL_PRES = find_column(weather, ["pres", "pressure", "slp"], required=False)
COL_TSUN = find_column(weather, ["tsun", "sunshine_minutes", "sun_minutes"], required=False)

weather_std = weather[[c for c in ["store_id", "date", COL_TAVG, COL_TMIN, COL_TMAX, COL_PRCP, COL_SNOW, COL_WSPD, COL_WDIR, COL_PRES, COL_TSUN] if c is not None]].copy()
weather_std = weather_std.rename(columns={
    COL_TAVG: "tavg",
    COL_TMIN: "tmin",
    COL_TMAX: "tmax",
    COL_PRCP: "prcp",
    COL_SNOW: "snow",
    COL_WSPD: "wspd",
    COL_WDIR: "wdir",
    COL_PRES: "pres",
    COL_TSUN: "tsun",
})

for col in ["tavg", "tmin", "tmax", "prcp", "snow", "wspd", "wdir", "pres", "tsun"]:
    if col not in weather_std.columns:
        weather_std[col] = np.nan

weather_std = ensure_numeric(weather_std, ["tavg", "tmin", "tmax", "prcp", "snow", "wspd", "wdir", "pres", "tsun"])
weather_std["temp_range"] = weather_std["tmax"] - weather_std["tmin"]

weather_std.head()

,store_id,date,tavg,tmin,tmax,prcp,snow,wspd,wdir,pres,tsun,temp_range
0,100500,2018-01-02,-10.3,-14.9,-2.7,0.0,0.0,7.9,NaN,NaN,NaN,12.2
1,100500,2018-01-03,-5.0,-8.8,2.8,0.0,0.0,9.4,NaN,NaN,NaN,11.6
2,100500,2018-01-04,-6.3,-11.6,-4.3,0.0,0.0,13.7,NaN,NaN,NaN,7.3
3,100500,2018-01-05,-6.2,-9.3,-0.5,0.0,0.0,9.4,NaN,NaN,NaN,8.8
4,100500,2018-01-06,-5.7,-11.0,-1.0,0.0,0.0,8.6,NaN,NaN,NaN,10.0


In [18]:
# Normalize key ids/dates first
perf = perf.copy()
perf = standardize_store_id_col(perf)
perf = standardize_date_col(perf, ["invoice_date", "date", "day"])

store_info = store_info.copy()
store_info = standardize_store_id_col(store_info)

# Keep only useful store columns.
lat_col = find_column(store_info, ["store_latitude", "latitude", "lat"], required=True)
lon_col = find_column(store_info, ["store_longitude", "longitude", "lon", "lng"], required=True)
region_col = find_column(store_info, ["region", "region_name", "region_id"], required=False)
area_col = find_column(store_info, ["area", "area_name", "area_id", "dma", "market_id"], required=False)
state_col = find_column(store_info, ["state", "store_state", "state_code"], required=False)
city_col = find_column(store_info, ["city", "store_city"], required=False)

store_cols = ["store_id", lat_col, lon_col]
for extra in [region_col, area_col, state_col, city_col]:
    if extra is not None and extra not in store_cols:
        store_cols.append(extra)

stores = store_info[store_cols].drop_duplicates(subset=["store_id"]).copy()
stores = stores.rename(columns={lat_col: "latitude", lon_col: "longitude"})
if region_col is not None:
    stores = stores.rename(columns={region_col: "region"})
if area_col is not None:
    stores = stores.rename(columns={area_col: "area"})
if state_col is not None:
    stores = stores.rename(columns={state_col: "state"})
if city_col is not None:
    stores = stores.rename(columns={city_col: "city"})

panel = perf.merge(stores, on="store_id", how="left")
panel = panel.merge(weather_std, on=["store_id", "date"], how="left")
panel = panel.sort_values(["store_id", "date"]).reset_index(drop=True)

for c in ["invoice_count", "oc_count", "fleet_oc_count"]:
    if c in panel.columns:
        panel[c] = pd.to_numeric(panel[c], errors="coerce")

print("panel shape:", panel.shape)
print("duplicate store-date rows:", panel.duplicated(subset=["store_id", "date"]).sum())
print("missing weather rates:")
print(panel[["tavg", "prcp", "snow"]].isna().mean())

panel.head()

panel shape: (773266, 21)
duplicate store-date rows: 0
missing weather rates:
tavg    0.023150
prcp    0.133078
snow    0.347084
dtype: float64


,date,store_id,invoice_count,oc_count,fleet_oc_count,latitude,longitude,region,area,state,city,tavg,tmin,tmax,prcp,snow,wspd,wdir,pres,tsun,temp_range
0,2018-01-02,79609,47,39,2,38.01075,-84.45542,79606,228371,KY,Lexington,-13.8,-19.3,-6.6,0.0,0.0,5.4,NaN,NaN,NaN,12.7
1,2018-01-03,79609,46,40,8,38.01075,-84.45542,79606,228371,KY,Lexington,-9.3,-15.5,-0.5,0.0,0.0,13.7,NaN,NaN,NaN,15.0
2,2018-01-04,79609,43,38,6,38.01075,-84.45542,79606,228371,KY,Lexington,-7.9,-13.2,-4.9,0.0,0.0,16.9,NaN,NaN,NaN,8.3
3,2018-01-05,79609,45,30,2,38.01075,-84.45542,79606,228371,KY,Lexington,-11.1,-14.9,-7.1,0.0,0.0,9.4,NaN,NaN,NaN,7.8
4,2018-01-06,79609,39,32,0,38.01075,-84.45542,79606,228371,KY,Lexington,-12.3,-18.8,-5.5,0.0,0.0,4.0,NaN,NaN,NaN,13.3


In [19]:
assert "store_id" in panel.columns
assert "date" in panel.columns
assert TARGET_COL in panel.columns, f"TARGET_COL={TARGET_COL!r} not present"

print("Date range:", panel["date"].min(), "to", panel["date"].max())
print("Stores:", panel["store_id"].nunique())
print("Rows:", len(panel))

n_dupes = panel.duplicated(subset=["store_id", "date"]).sum()
print("Duplicate store-date rows:", n_dupes)
if n_dupes > 0:
    raise ValueError("Duplicate store-date rows detected. Fix this before feature engineering.")

print(f"Missing rate for {TARGET_COL}:", panel[TARGET_COL].isna().mean())

weather_check_cols = [c for c in ["tavg", "tmin", "tmax", "prcp", "snow", "wspd"] if c in panel.columns]
if weather_check_cols:
    print("\nWeather missingness:")
    print(panel[weather_check_cols].isna().mean().sort_values())

coverage = panel.groupby("store_id")["date"].agg(["min", "max", "count"]).reset_index()
print("\nCoverage preview:")
display(coverage.head())

print("\nCoverage count summary:")
print(coverage["count"].describe())

Date range: 2018-01-02 00:00:00 to 2022-12-31 00:00:00
Stores: 439
Rows: 773266
Duplicate store-date rows: 0
Missing rate for oc_count: 0.0

Weather missingness:
tavg    0.023150
wspd    0.026409
tmax    0.027039
tmin    0.027040
prcp    0.133078
snow    0.347084
dtype: float64

Coverage preview:


,store_id,min,max,count
0,79609,2018-01-02,2022-12-31,1789
1,81958,2018-01-02,2022-12-31,1791
2,83444,2018-01-02,2022-12-31,1791
3,84321,2018-01-02,2022-12-31,1789
4,84328,2018-01-02,2022-12-31,1788



Coverage count summary:
count     439.000000
mean     1761.425968
std        72.804601
min      1373.000000
25%      1783.500000
50%      1789.000000
75%      1791.000000
max      1792.000000
Name: count, dtype: float64


In [ ]:
panel["is_rain"] = (panel["prcp"].fillna(0) > 0).astype(int)
panel["is_snow"] = (panel["snow"].fillna(0) > 0).astype(int)
panel["is_freezing"] = (panel["tavg"] <= 0).astype(int)
panel["is_very_cold"] = ((panel["tavg"] > 0) & (panel["tavg"] <= 7)).astype(int)
panel["is_hot"] = ((panel["tavg"] >= 27) & (panel["tavg"] <= 35)).astype(int)
panel["is_heavy_rain"] = (panel["prcp"].fillna(0) >= 10).astype(int)
panel["is_heavy_snow"] = (panel["snow"].fillna(0) >= 5).astype(int)

# Weather bucket priority matters. We assign the strongest bucket first.
def weather_bucket(row: pd.Series) -> str:
    if row["is_heavy_rain"] == 1:
        return "Heavy Rain"
    if row["is_heavy_snow"] == 1:
        return "Heavy Snow"
    if row["is_snow"] == 1:
        return "Any Snow"
    if row["is_rain"] == 1:
        return "Light Rain"
    if row["is_freezing"] == 1:
        return "Freezing (≤0°C)"
    if row["is_hot"] == 1:
        return "Hot (27–35°C)"
    if row["is_very_cold"] == 1:
        return "Very Cold (0–7°C)"
    return BASELINE_LABEL

panel["weather_condition"] = panel.apply(weather_bucket, axis=1)
panel[["date", "store_id", "tavg", "prcp", "snow", "weather_condition"]].head(10)

,date,store_id,tavg,prcp,snow,weather_condition
0,2018-01-02,79609,-13.8,0.0,0.0,Freezing (≤0°C)
1,2018-01-03,79609,-9.3,0.0,0.0,Freezing (≤0°C)
2,2018-01-04,79609,-7.9,0.0,0.0,Freezing (≤0°C)
3,2018-01-05,79609,-11.1,0.0,0.0,Freezing (≤0°C)
4,2018-01-06,79609,-12.3,0.0,0.0,Freezing (≤0°C)
5,2018-01-07,79609,-6.4,0.0,0.0,Freezing (≤0°C)
6,2018-01-08,79609,2.6,7.1,0.0,Light Rain
7,2018-01-09,79609,3.9,0.0,0.0,Very Cold (0–7°C)
8,2018-01-10,79609,10.2,0.0,0.0,Clear / Mild (baseline)
9,2018-01-11,79609,14.0,2.3,0.0,Light Rain


In [21]:
WEATHER_LAG_BASE_COLS = ["tavg", "tmin", "tmax", "temp_range", "prcp", "snow", "wspd", "pres", "tsun", "is_rain", "is_snow", "is_freezing", "is_heavy_rain", "is_heavy_snow"]
WEATHER_LAGS = [1, 2, 3]

panel = panel.sort_values(["store_id", "date"]).reset_index(drop=True)
for base_col in WEATHER_LAG_BASE_COLS:
    if base_col not in panel.columns:
        continue
    for lag in WEATHER_LAGS:
        panel[f"{base_col}_lag{lag}"] = panel.groupby("store_id", sort=False)[base_col].shift(lag)

lag_preview_cols = [c for c in panel.columns if c.endswith("_lag1")][:12]
panel[["store_id", "date"] + lag_preview_cols].head(10)

,store_id,date,tavg_lag1,tmin_lag1,tmax_lag1,temp_range_lag1,prcp_lag1,snow_lag1,wspd_lag1,pres_lag1,tsun_lag1,is_rain_lag1,is_snow_lag1,is_freezing_lag1
0,79609,2018-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,79609,2018-01-03,-13.8,-19.3,-6.6,12.7,0.0,0.0,5.4,NaN,NaN,0.0,0.0,1.0
2,79609,2018-01-04,-9.3,-15.5,-0.5,15.0,0.0,0.0,13.7,NaN,NaN,0.0,0.0,1.0
3,79609,2018-01-05,-7.9,-13.2,-4.9,8.3,0.0,0.0,16.9,NaN,NaN,0.0,0.0,1.0
4,79609,2018-01-06,-11.1,-14.9,-7.1,7.8,0.0,0.0,9.4,NaN,NaN,0.0,0.0,1.0
5,79609,2018-01-07,-12.3,-18.8,-5.5,13.3,0.0,0.0,4.0,NaN,NaN,0.0,0.0,1.0
6,79609,2018-01-08,-6.4,-12.1,5.6,17.7,0.0,0.0,17.3,NaN,NaN,0.0,0.0,1.0
7,79609,2018-01-09,2.6,1.7,3.3,1.6,7.1,0.0,12.2,NaN,NaN,1.0,0.0,0.0
8,79609,2018-01-10,3.9,1.7,8.3,6.6,0.0,0.0,7.2,NaN,NaN,0.0,0.0,0.0
9,79609,2018-01-11,10.2,4.4,18.3,13.9,0.0,0.0,17.6,NaN,NaN,0.0,0.0,0.0


In [ ]:
def add_days_since_event(
    df: pd.DataFrame,
    event_col: str,
    group_col: str = "store_id",
    date_col: str = "date",
    out_col: Optional[str] = None,
) -> pd.Series:
    """
    For each row, compute number of days since the last row where event_col==1 within the same store.
    If no prior event exists, result is NaN.
    """
    if out_col is None:
        out_col = f"days_since_{event_col}"

    df = df.sort_values([group_col, date_col]).copy()
    result = pd.Series(index=df.index, dtype="float64")

    for _, g in df.groupby(group_col, sort=False):
        dates = pd.to_datetime(g[date_col]).values.astype("datetime64[D]")
        events = g[event_col].fillna(0).astype(int).values
        out = np.full(len(g), np.nan)
        last_event_date = None
        for i in range(len(g)):
            if last_event_date is not None:
                out[i] = (dates[i] - last_event_date).astype(int)
            if events[i] == 1:
                last_event_date = dates[i]
        result.loc[g.index] = out
    return result

panel["prcp_roll3_mean"] = (
    panel.groupby("store_id", sort=False)["prcp"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
)

panel["days_since_last_rain"] = add_days_since_event(panel, event_col="is_rain")
panel["days_since_last_snow"] = add_days_since_event(panel, event_col="is_snow")
panel["days_since_last_rain"] = (
    panel.groupby("store_id")["days_since_last_rain"].shift(1)
)
panel["days_since_last_snow"] = (
    panel.groupby("store_id")["days_since_last_snow"].shift(1)
)
panel[[
    "store_id", "date", "prcp", "prcp_roll3_mean", "is_rain", "days_since_last_rain", "is_snow", "days_since_last_snow"
]].head(12)

,store_id,date,prcp,prcp_roll3_mean,is_rain,days_since_last_rain,is_snow,days_since_last_snow
0,79609,2018-01-02,0.0,NaN,0,NaN,0,NaN
1,79609,2018-01-03,0.0,0.000000,0,NaN,0,NaN
2,79609,2018-01-04,0.0,0.000000,0,NaN,0,NaN
3,79609,2018-01-05,0.0,0.000000,0,NaN,0,NaN
4,79609,2018-01-06,0.0,0.000000,0,NaN,0,NaN
5,79609,2018-01-07,0.0,0.000000,0,NaN,0,NaN
6,79609,2018-01-08,7.1,0.000000,1,NaN,0,NaN
7,79609,2018-01-09,0.0,2.366667,0,1.0,0,NaN
8,79609,2018-01-10,0.0,2.366667,0,2.0,0,NaN
9,79609,2018-01-11,2.3,2.366667,1,3.0,0,NaN


In [23]:
store_geo = stores[["store_id", "latitude", "longitude"]].dropna().drop_duplicates().copy()
store_geo["latitude_rad"] = np.deg2rad(store_geo["latitude"].astype(float))
store_geo["longitude_rad"] = np.deg2rad(store_geo["longitude"].astype(float))

if not SKLEARN_AVAILABLE:
    raise ImportError("scikit-learn is required for the nearby store map (BallTree).")

coords_rad = store_geo[["latitude_rad", "longitude_rad"]].to_numpy()
tree = BallTree(coords_rad, metric="haversine")

earth_radius_km = 6371.0088
k_query = min(NEIGHBOR_K + 1, len(store_geo))  # +1 because the first match is the store itself

dist_rad, ind = tree.query(coords_rad, k=k_query)
dist_km = dist_rad * earth_radius_km

neighbor_rows = []
for i, store_id in enumerate(store_geo["store_id"].tolist()):
    rank = 0
    for j, d_km in zip(ind[i], dist_km[i]):
        nbr_store = store_geo.iloc[j]["store_id"]
        if nbr_store == store_id:
            continue
        if d_km > MAX_NEIGHBOR_DISTANCE_KM:
            continue
        rank += 1
        neighbor_rows.append({
            "store_id": store_id,
            "neighbor_store_id": nbr_store,
            "neighbor_rank": rank,
            "distance_km": float(d_km),
        })
        if rank >= NEIGHBOR_K:
            break

neighbor_map = pd.DataFrame(neighbor_rows)
neighbor_map.to_csv(OUT_NEIGHBOR_MAP_PATH, index=False)
print(f"[INFO] nearby store map saved -> {OUT_NEIGHBOR_MAP_PATH}")
print(neighbor_map.head(10))

# %%
# Bring in neighbor demand by date.
demand_small = panel[["store_id", "date", TARGET_COL]].copy()
demand_small = demand_small.rename(columns={"store_id": "neighbor_store_id", TARGET_COL: f"neighbor_{TARGET_COL}"})

neighbor_daily = neighbor_map.merge(demand_small, on="neighbor_store_id", how="left")
neighbor_daily = neighbor_daily.groupby(["store_id", "date"], as_index=False).agg(
    regional_neighbor_demand_mean=(f"neighbor_{TARGET_COL}", "mean"),
    regional_neighbor_demand_median=(f"neighbor_{TARGET_COL}", "median"),
    regional_neighbor_store_count=(f"neighbor_{TARGET_COL}", "count"),
)

panel = panel.merge(neighbor_daily, on=["store_id", "date"], how="left")

# Safe forecasting versions: lag the regional index.
panel["regional_neighbor_demand_mean_lag1"] = panel.groupby("store_id", sort=False)["regional_neighbor_demand_mean"].shift(1)
panel["regional_neighbor_demand_mean_lag3"] = panel.groupby("store_id", sort=False)["regional_neighbor_demand_mean"].shift(3)
panel["regional_neighbor_demand_mean_roll7_lag1"] = (
    panel.groupby("store_id", sort=False)["regional_neighbor_demand_mean"]
    .transform(lambda s: s.shift(1).rolling(7, min_periods=3).mean())
)

panel[[
    "store_id", "date", TARGET_COL,
    "regional_neighbor_demand_mean", "regional_neighbor_demand_mean_lag1", "regional_neighbor_demand_mean_roll7_lag1"
]].head(10)

[INFO] nearby store map saved -> /Users/harshini/GenAI-Weather-Based-Store-Analytics/data_processed/nearby_store_map_top5.csv
   store_id  neighbor_store_id  neighbor_rank  distance_km
0     79609            83444.0              1     0.613188
1     79609            81958.0              2     3.306583
2     79609            84321.0              3     5.413689
3     79609           602320.0              4     5.975437
4     79609            84328.0              5     6.233292
5     84321            84328.0              1     4.621868
6     84321            84329.0              2     4.918579
7     84321            79609.0              3     5.413689
8     84321            83444.0              4     5.754746
9     84321           602320.0              5     6.144767


,store_id,date,oc_count,regional_neighbor_demand_mean,regional_neighbor_demand_mean_lag1,regional_neighbor_demand_mean_roll7_lag1
0,79609,2018-01-02,39,62.0,NaN,NaN
1,79609,2018-01-03,40,58.8,62.0,NaN
2,79609,2018-01-04,38,55.0,58.8,NaN
3,79609,2018-01-05,30,54.4,55.0,58.600000
4,79609,2018-01-06,32,55.4,54.4,57.550000
5,79609,2018-01-07,22,48.0,55.4,57.120000
6,79609,2018-01-08,32,41.4,48.0,55.600000
7,79609,2018-01-09,37,42.2,41.4,53.571429
8,79609,2018-01-10,41,52.8,42.2,50.742857
9,79609,2018-01-11,37,51.6,52.8,49.885714


In [24]:
panel["year"] = panel["date"].dt.year
panel["month"] = panel["date"].dt.month
panel["dayofweek"] = panel["date"].dt.dayofweek
panel["is_weekend"] = (panel["dayofweek"] >= 5).astype(int)
panel["dayofyear"] = panel["date"].dt.dayofyear

# %% [markdown]
# ## 11) Final feature sanity checks

# %%
new_feature_cols = [
    # weather lags
    *[f"{c}_lag{lag}" for c in WEATHER_LAG_BASE_COLS if c in panel.columns for lag in WEATHER_LAGS],
    # hangover
    "prcp_roll3_mean",
    "days_since_last_rain",
    "days_since_last_snow",
    # nearby demand
    "regional_neighbor_demand_mean",
    "regional_neighbor_demand_median",
    "regional_neighbor_store_count",
    "regional_neighbor_demand_mean_lag1",
    "regional_neighbor_demand_mean_lag3",
    "regional_neighbor_demand_mean_roll7_lag1",
]
new_feature_cols = [c for c in new_feature_cols if c in panel.columns]

missing_pct = panel[new_feature_cols].isna().mean().sort_values(ascending=False)
print(missing_pct.head(30))


tsun_lag2               1.000000
tsun_lag3               1.000000
tsun_lag1               1.000000
pres_lag3               1.000000
pres_lag2               1.000000
pres_lag1               1.000000
days_since_last_snow    0.374087
snow_lag3               0.348178
snow_lag2               0.347813
snow_lag1               0.347448
prcp_lag3               0.134669
prcp_lag2               0.134150
prcp_lag1               0.133628
prcp_roll3_mean         0.075878
days_since_last_rain    0.056503
tmin_lag3               0.028724
temp_range_lag3         0.028724
tmax_lag3               0.028722
temp_range_lag2         0.028162
tmin_lag2               0.028162
tmax_lag2               0.028161
wspd_lag3               0.028093
temp_range_lag1         0.027601
tmin_lag1               0.027601
tmax_lag1               0.027600
wspd_lag2               0.027531
wspd_lag1               0.026970
tavg_lag3               0.024834
tavg_lag2               0.024272
tavg_lag1               0.023711
dtype: flo

In [25]:
panel.to_csv(OUT_FEATURES_PATH, index=False)
print(f"[INFO] feature table saved -> {OUT_FEATURES_PATH}")
print("shape:", panel.shape)

[INFO] feature table saved -> /Users/harshini/GenAI-Weather-Based-Store-Analytics/data_processed/sprint3_features_weather_impact_ready.csv
shape: (773266, 85)


In [ ]:
panel["dayofweek"] = panel["date"].dt.dayofweek
panel["month"]     = panel["date"].dt.month
impact_df = panel.loc[panel["date"] >= VALID_START, ["date", "store_id", TARGET_COL, "weather_condition"]].copy()
impact_df = impact_df.dropna(subset=[TARGET_COL])

# Compute store-level baseline (clear/mild days per store)
baseline_by_store = (
    impact_df[impact_df["weather_condition"] == BASELINE_LABEL]
    .groupby(["store_id", "dayofweek", "month"])[TARGET_COL]
    .mean()
    .rename("store_baseline")
    .reset_index()
)
impact_df = impact_df.merge(
    baseline_by_store,
    on=["store_id", "dayofweek", "month"],
    how="left"
)

# Drop rows where baseline is missing (rare but possible)
impact_df = impact_df.dropna(subset=["store_baseline"])

# Compute % change vs store-specific baseline
impact_df["pct_vs_baseline"] = (
    (impact_df[TARGET_COL] - impact_df["store_baseline"])
    / impact_df["store_baseline"]
) * 100

rows = []
ordered_conditions = [
    "Heavy Rain",
    "Heavy Snow",
    "Any Snow",
    "Light Rain",
    "Freezing (≤0°C)",
    "Hot (27–35°C)",
    BASELINE_LABEL,
    "Very Cold (0–7°C)",
]

seen_conditions = set(impact_df["weather_condition"].dropna().unique().tolist())
ordered_conditions = [c for c in ordered_conditions if c in seen_conditions] + [
    c for c in sorted(seen_conditions) if c not in ordered_conditions
]

store_effects = (
    impact_df.groupby(["store_id", "weather_condition"])["pct_vs_baseline"]
    .mean()
    .reset_index()
)

for cond in ordered_conditions:
    g = store_effects.loc[store_effects["weather_condition"] == cond, "pct_vs_baseline"]    
    mean_pct, ci_low, ci_high = mean_ci_from_series(g)
    rows.append({
        "Weather Condition": cond,
        "N Days": int(g.notna().sum()),
        "Mean % vs Baseline": round(mean_pct, 3) if pd.notna(mean_pct) else np.nan,
        "95% CI Low": round(ci_low, 3) if pd.notna(ci_low) else np.nan,
        "95% CI High": round(ci_high, 3) if pd.notna(ci_high) else np.nan,
        "Significant?": "YES" if pd.notna(ci_low) and pd.notna(ci_high) and not (ci_low <= 0 <= ci_high) else "NO",
    })

weather_impact_table = pd.DataFrame(rows)
weather_impact_table.to_csv(OUT_WEATHER_IMPACT_PATH, index=False)
print(f"[INFO] weather impact table saved -> {OUT_WEATHER_IMPACT_PATH}")
weather_impact_table

[INFO] weather impact table saved -> /Users/harshini/GenAI-Weather-Based-Store-Analytics/data_processed/weather_impact_ci_2022_sprint3.csv


,Weather Condition,N Days,Mean % vs Baseline,95% CI Low,95% CI High,Significant?
0,Heavy Rain,431,-10.232,-11.027,-9.438,YES
1,Heavy Snow,239,-18.103,-19.602,-16.604,YES
2,Any Snow,240,-13.615,-15.481,-11.750,YES
3,Light Rain,434,-3.673,-4.075,-3.272,YES
4,Freezing (≤0°C),430,-7.997,-9.131,-6.863,YES
5,Hot (27–35°C),411,1.807,0.735,2.879,YES
6,Clear / Mild (baseline),439,0.000,-0.000,0.000,NO
7,Very Cold (0–7°C),434,-2.818,-3.320,-2.317,YES


In [30]:
def build_weather_impact_summary(df: pd.DataFrame) -> str:
    if df.empty:
        return "No weather impact results were generated."

    sig = df[df["Significant?"] == "YES"].copy()
    if sig.empty:
        return (
            f"Using {TARGET_COL} in the 2022 validation window, none of the tested weather buckets "
            "showed a 95% confidence interval fully away from zero relative to the clear/mild baseline."
        )

    worst = sig.sort_values("Mean % vs Baseline").iloc[0]
    best = sig.sort_values("Mean % vs Baseline", ascending=False).iloc[0]

    return (
        f"Using {TARGET_COL} in the 2022 validation window, the weather impact table shows that several weather "
        f"conditions are materially different from the clear/mild baseline. The strongest negative effect is "
        f"{worst['Weather Condition']} at {worst['Mean % vs Baseline']}% vs baseline "
        f"(95% CI {worst['95% CI Low']} to {worst['95% CI High']}), while the strongest positive effect is "
        f"{best['Weather Condition']} at {best['Mean % vs Baseline']}% "
        f"(95% CI {best['95% CI Low']} to {best['95% CI High']})."
    )

summary_text = build_weather_impact_summary(weather_impact_table)
print(summary_text)

Using oc_count in the 2022 validation window, the weather impact table shows that several weather conditions are materially different from the clear/mild baseline. The strongest negative effect is Heavy Snow at -18.103% vs baseline (95% CI -19.602 to -16.604), while the strongest positive effect is Hot (27–35°C) at 1.807% (95% CI 0.735 to 2.879).
